In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import pandas as pd
import numpy as np
import glob, os, pickle, warnings
warnings.filterwarnings('ignore')

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CELL 2 — Carga de datos
# Cada fila del CSV está envuelta en comillas dobles con ; como separador interno.
# ============================================================
DATA_PATH = "/kaggle/input/datasets/axelnahuelbelbrun/subtes"

def read_molinetes_csv(filepath):
    with open(filepath, 'r', encoding='utf-8-sig') as f:
        lines = f.read().strip().split('\n')
    rows = [line.strip().strip('"').split(';') for line in lines if line.strip()]
    header = [h.strip() for h in rows[0]]
    n_cols = sum(1 for h in header if h)
    data = [row[:n_cols] for row in rows[1:] if len(row) >= n_cols]
    return pd.DataFrame(data, columns=header[:n_cols])

files = glob.glob(os.path.join(DATA_PATH, "*.csv"))
print(f"Archivos: {len(files)}")

raw = pd.concat([read_molinetes_csv(f) for f in files], ignore_index=True)

raw['FECHA']     = pd.to_datetime(raw['FECHA'], dayfirst=True, errors='coerce')
raw['DESDE']     = pd.to_timedelta(raw['DESDE'], errors='coerce')
raw['pax_TOTAL'] = pd.to_numeric(raw['pax_TOTAL'], errors='coerce')
raw['TIMESTAMP'] = raw['FECHA'] + raw['DESDE']
raw['ESTACION']  = raw['ESTACION'].str.strip()
raw['LINEA']     = raw['LINEA'].str.strip()
raw = raw.dropna(subset=['TIMESTAMP', 'pax_TOTAL', 'LINEA', 'ESTACION'])
raw['pax_TOTAL'] = raw['pax_TOTAL'].astype(int)

print(f"Filas totales: {len(raw):,}")
raw.head()

In [ ]:
# ============================================================
# CELL 3 — Agregar por estación (múltiples molinetes → 1 fila)
# ============================================================
agg = (
    raw
    .groupby(['TIMESTAMP', 'LINEA', 'ESTACION'], as_index=False)['pax_TOTAL']
    .sum()
    .rename(columns={'pax_TOTAL': 'boardings'})
)
print(f"Filas agregadas: {len(agg):,}")

In [ ]:
# ============================================================
# CELL 4 — DIAGNÓSTICO: nombres exactos de estaciones por línea
# Verificar que coincidan con las secuencias del Cell 5.
# ============================================================
for linea in sorted(agg['LINEA'].unique()):
    sts = sorted(agg[agg['LINEA'] == linea]['ESTACION'].unique())
    print(f"\n{linea}  ({len(sts)} estaciones):")
    for s in sts:
        print(f"  '{s}'")

In [ ]:
# ============================================================
# CELL 5 — Secuencias de estaciones + optimización automática de k
#
# PARÁMETROS AJUSTABLES:
#   STATION_SEQUENCES : orden de estaciones por línea (terminal A → terminal B).
#                       Debe coincidir exactamente con los nombres del CSV.
#
#   k (longitud promedio de viaje en estaciones): se optimiza automáticamente
#     por línea buscando el mayor k tal que la carga residual en el terminal B
#     sea < TERMINAL_SCORE_THRESHOLD del pico de carga.
#     Se puede sobreescribir manualmente al final de la celda.
#
#   TERMINAL_SCORE_THRESHOLD : fracción máxima de carga en terminal B respecto
#     al pico. Valores típicos: 0.10-0.25.
# ============================================================

STATION_SEQUENCES = {
    'LineaA': [
        'Plaza de Mayo', 'Peru', 'Piedras', 'Lima', 'Saenz Peña',
        'Congreso', 'Pasco', 'Alberti', 'Plaza Miserere', 'Loria',
        'Castro Barros', 'Rio de Janeiro', 'Acoyte', 'Primera Junta',
        'Puan', 'Carabobo', 'Flores', 'San Pedrito',
    ],
    'LineaB': [
        'Leandro N. Alem', 'Florida', 'Carlos Pellegrini', 'Uruguay',
        'Callao.B', 'Pasteur', 'Pueyrredon', 'Carlos Gardel', 'Medrano',
        'Angel Gallardo', 'Malabia', 'Dorrego', 'Federico Lacroze',
        'Tronador', 'Los Incas', 'Echeverria', 'Rosas',
    ],
    'LineaC': [
        'Constitucion', 'San Juan', 'Independencia', 'Mariano Moreno',
        'Avenida de Mayo', 'Diagonal Norte', 'Lavalle',
        'General San Martin', 'Retiro',
    ],
    'LineaD': [
        'Catedral', '9 de julio', 'Tribunales', 'Callao',
        'Facultad de Medicina', 'Pueyrredon.D', 'Agüero', 'Bulnes',
        'Scalabrini Ortiz', 'Plaza Italia', 'Palermo', 'Ministro Carranza',
        'Olleros', 'Jose Hernandez', 'Juramento', 'Congreso de Tucuman',
    ],
    'LineaE': [
        'Pza. de los Virreyes', 'Varela', 'Medalla Milagrosa',
        'Emilio Mitre', 'Jose Maria Moreno', 'Avenida La Plata', 'Boedo',
        'Urquiza', 'Jujuy', 'Pichincha', 'Entre Rios', 'San Jose',
        'Independencia.H', 'General Belgrano', 'Bolivar',
        'Correo Central', 'Catalinas', 'Retiro E',
    ],
    'LineaH': [
        'Hospitales', 'Patricios', 'Caseros', 'Inclan', 'Humberto I',
        'Venezuela', 'Once', 'Corrientes', 'Cordoba', 'Santa Fe',
        'Las Heras', 'Facultad de Derecho',
    ],
}

seq_pos = {
    linea: {st: i for i, st in enumerate(seq)}
    for linea, seq in STATION_SEQUENCES.items()
}

agg['station_pos'] = agg.apply(
    lambda r: seq_pos.get(r['LINEA'], {}).get(r['ESTACION'], -1), axis=1
)
agg['n_stations'] = agg['LINEA'].map(lambda l: len(STATION_SEQUENCES.get(l, [])))

unmatched = agg[agg['station_pos'] == -1][['LINEA', 'ESTACION']].drop_duplicates()
print(f"Estaciones sin mapear: {len(unmatched)}")
if len(unmatched):
    print(unmatched.sort_values(['LINEA', 'ESTACION']).to_string(index=False))

agg_valid = agg[agg['station_pos'] >= 0].copy()

# --- Optimización automática de k por línea ---
# Score = carga_media_terminal_B / pico_carga_media. Queremos score < threshold.
# Elegimos el mayor k que aún cumple la condición (más cobertura, menos residual).
TERMINAL_SCORE_THRESHOLD = 0.20

best_k = {}
print("\nOptimización de k (longitud promedio de viaje en estaciones):")
print(f"{'Línea':<12} {'k_opt':>5}  {'score_terminal':>14}  {'carga_media_pico':>16}")
print("-" * 55)

for linea in sorted(agg_valid['LINEA'].unique()):
    sub  = agg_valid[agg_valid['LINEA'] == linea]
    n_st = len(STATION_SEQUENCES.get(linea, []))
    if n_st < 4:
        best_k[linea] = 2
        continue

    pivot = (
        sub.pivot_table(index='TIMESTAMP', columns='station_pos',
                        values='boardings', fill_value=0)
        .sort_index(axis=1)
    )

    scores = {}
    for k in range(2, min(n_st - 1, 11)):
        lp = pd.DataFrame(0, index=pivot.index, columns=pivot.columns)
        for idx, col in enumerate(pivot.columns):
            start = max(0, idx - k)
            if idx > start:
                lp[col] = pivot.iloc[:, start:idx].sum(axis=1)
        mean_profile = lp.mean()
        peak         = mean_profile.max()
        terminal_B   = mean_profile.iloc[-1]
        scores[k]    = (terminal_B / (peak + 1e-9), peak)

    eligible = {k: v for k, v in scores.items() if v[0] < TERMINAL_SCORE_THRESHOLD}
    if eligible:
        chosen_k = max(eligible.keys())
    else:
        chosen_k = min(scores, key=lambda k: scores[k][0])

    best_k[linea] = chosen_k
    score, peak = scores[chosen_k]
    print(f"{linea:<12} {chosen_k:>5}  {score:>14.3f}  {peak:>16.1f}")

# --- Override manual (descomentar si se desea ajustar una línea) ---
# best_k['LineaA'] = 6
# best_k['LineaC'] = 4

print("\nk definitivos:", best_k)

In [ ]:
# ============================================================
# CELL 6 — Estimación de carga (load)
#
# load(i, t) = sum(boardings en estaciones [i-k .. i-1], mismo t)
# Terminal constraint implícito: carga en estación 0 = 0. ✓
# ============================================================
load_parts = []

for linea in agg_valid['LINEA'].unique():
    k   = best_k.get(linea, 5)
    sub = agg_valid[agg_valid['LINEA'] == linea]

    pivot = (
        sub.pivot_table(index='TIMESTAMP', columns='station_pos',
                        values='boardings', fill_value=0)
        .sort_index(axis=1)
    )

    load_pivot = pd.DataFrame(0, index=pivot.index, columns=pivot.columns)
    for idx, col in enumerate(pivot.columns):
        start = max(0, idx - k)
        if idx > start:
            load_pivot[col] = pivot.iloc[:, start:idx].sum(axis=1)

    load_long = (
        load_pivot.stack()
        .reset_index()
        .rename(columns={0: 'load_est'})
    )
    load_long['LINEA'] = linea
    load_parts.append(load_long)

load_estimates = pd.concat(load_parts, ignore_index=True)
load_df = agg_valid.merge(load_estimates, on=['LINEA', 'TIMESTAMP', 'station_pos'], how='left')
load_df['load_est'] = load_df['load_est'].fillna(0).astype(int)

print(load_df[['LINEA', 'ESTACION', 'station_pos', 'boardings', 'load_est']].head(20))

In [ ]:
# ============================================================
# CELL 7 — Feature engineering
#
# PARÁMETROS AJUSTABLES:
#   LAGS  : intervalos de 15 min hacia atrás en el tiempo.
#            96 = 1 día completo. Agregar 672 para incluir 1 semana atrás.
#   ROLLS : ventanas de media móvil (en intervalos de 15 min).
# ============================================================
LAGS  = [1, 2, 4, 8, 96]   # 15m, 30m, 1h, 2h, 1 día
ROLLS = [4, 8, 96]          # 1h, 2h, 1 día

df = load_df.copy()

# --- Features temporales (sin dependencia de orden entre filas) ---
df['hour']         = df['TIMESTAMP'].dt.hour
df['minute']       = df['TIMESTAMP'].dt.minute
df['dayofweek']    = df['TIMESTAMP'].dt.dayofweek
df['month']        = df['TIMESTAMP'].dt.month
df['is_weekend']   = (df['dayofweek'] >= 5).astype(int)
df['time_bin']     = df['hour'] * 4 + df['minute'] // 15
df['station_norm'] = df['station_pos'] / df['n_stations'].clip(lower=1)
df['linea_enc']    = pd.Categorical(df['LINEA']).codes
df['estacion_enc'] = pd.Categorical(df['ESTACION']).codes

# --- Lags y rolling temporales: requieren orden (LINEA, ESTACION, TIMESTAMP) ---
df = df.sort_values(['LINEA', 'ESTACION', 'TIMESTAMP']).reset_index(drop=True)
grp_time = df.groupby(['LINEA', 'ESTACION'])

for lag in LAGS:
    df[f'b_lag_{lag}']    = grp_time['boardings'].shift(lag)
    df[f'load_lag_{lag}'] = grp_time['load_est'].shift(lag)

for win in ROLLS:
    df[f'b_roll_{win}'] = grp_time['boardings'].transform(
        lambda x: x.shift(1).rolling(win, min_periods=1).mean()
    )

# --- Features espaciales: requieren orden (LINEA, TIMESTAMP, station_pos) ---
# Capturan cuánta gente subió en las estaciones vecinas en el mismo intervalo,
# lo que es una señal directa de la presión de carga en el tramo.
df = df.sort_values(['LINEA', 'TIMESTAMP', 'station_pos']).reset_index(drop=True)
grp_space = df.groupby(['LINEA', 'TIMESTAMP'])

df['b_prev_1st'] = grp_space['boardings'].shift(1)   # estación inmediata anterior
df['b_prev_2nd'] = grp_space['boardings'].shift(2)   # dos estaciones atrás
df['b_next_1st'] = grp_space['boardings'].shift(-1)  # estación siguiente

FEATURES = [
    'hour', 'minute', 'dayofweek', 'month', 'is_weekend', 'time_bin',
    'station_pos', 'station_norm',
    'boardings',
    *[f'b_lag_{l}'    for l in LAGS],
    *[f'load_lag_{l}' for l in LAGS],
    *[f'b_roll_{w}'   for w in ROLLS],
    'b_prev_1st', 'b_prev_2nd', 'b_next_1st',
    'linea_enc', 'estacion_enc',
]
TARGET = 'load_est'

print(f"Features totales: {len(FEATURES)}")
print(FEATURES)

In [ ]:
# ============================================================
# CELL 8 — Entrenamiento XGBoost
#
# PARÁMETROS AJUSTABLES DEL MODELO:
#   n_estimators       : número máximo de árboles. El early stopping lo recorta.
#   learning_rate      : paso de aprendizaje. Más bajo = más árboles necesarios
#                        pero generalmente mejor generalización. Rango típico: 0.01-0.1.
#   max_depth          : profundidad máxima de cada árbol. Más alto = captura
#                        patrones más complejos pero riesgo de overfitting.
#                        Rango típico: 4-10.
#   min_child_weight   : mínimo de muestras en una hoja. Más alto = más
#                        conservador. Útil para evitar aprender de grupos pequeños.
#   subsample          : fracción de filas por árbol. Regularización.
#   colsample_bytree   : fracción de features por árbol. Regularización.
#   early_stopping_rounds: detiene si no mejora en N rondas consecutivas.
#
# PARÁMETRO DE SPLIT:
#   TEST_DAYS : días usados como test (split temporal, siempre los más recientes).
# ============================================================
TEST_DAYS = 30

split_date = df['TIMESTAMP'].max() - pd.Timedelta(days=TEST_DAYS)
train_df = df[df['TIMESTAMP'] < split_date].dropna(subset=FEATURES + [TARGET])
test_df  = df[df['TIMESTAMP'] >= split_date].dropna(subset=FEATURES + [TARGET])

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_test,  y_test  = test_df[FEATURES],  test_df[TARGET]
print(f"Train: {len(X_train):,}  |  Test: {len(X_test):,}")

model = XGBRegressor(
    n_estimators          = 800,
    learning_rate         = 0.05,
    max_depth             = 8,
    min_child_weight      = 5,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    tree_method           = 'hist',
    device                = 'cpu',
    early_stopping_rounds = 30,
    eval_metric           = 'mae',
    random_state          = 42,
    n_jobs                = -1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)

In [ ]:
# ============================================================
# CELL 9 — Evaluación
# ============================================================
y_pred = np.maximum(model.predict(X_test), 0)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mape = np.mean(np.abs((y_test - y_pred) / (y_test.clip(lower=1)))) * 100

# MAPE filtrado: excluye intervalos con carga casi nula (madrugada, terminales)
# donde el % se dispara por denominadores pequeños.
MAPE_MIN_LOAD = 20
mask = y_test > MAPE_MIN_LOAD
mape_filtrado = np.mean(
    np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])
) * 100

print(f"MAE            : {mae:.1f} pasajeros estimados")
print(f"RMSE           : {rmse:.1f}")
print(f"MAPE (total)   : {mape:.1f}%")
print(f"MAPE (load>{MAPE_MIN_LOAD}): {mape_filtrado:.1f}%")
print(f"Filas filtradas: {mask.sum():,} de {len(y_test):,} ({mask.mean()*100:.1f}%)")

# Feature importance
feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

feat_imp.head(15).plot(kind='barh', ax=axes[0], title='Feature Importance (top 15)')
axes[0].invert_yaxis()

# Dispersión real vs predicho (muestra aleatoria para no saturar el gráfico)
sample = np.random.choice(len(y_test), size=min(5000, len(y_test)), replace=False)
axes[1].scatter(y_test.iloc[sample], y_pred[sample], alpha=0.3, s=5)
lim = max(y_test.max(), y_pred.max())
axes[1].plot([0, lim], [0, lim], 'r--', linewidth=1)
axes[1].set_xlabel('load_est real')
axes[1].set_ylabel('load_est predicho')
axes[1].set_title('Real vs Predicho')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 10 — Guardar modelo
# En Kaggle el entorno es volátil; los archivos en /kaggle/working
# quedan disponibles como Output del notebook para descargar o reusar.
# ============================================================
OUTPUT_DIR = "/kaggle/working"

# Modelo en formato nativo XGBoost (portable entre versiones)
model_path = os.path.join(OUTPUT_DIR, "xgb_subte_load.json")
model.save_model(model_path)

# Metadata necesaria para reproducir inferencias fuera del notebook
meta = {
    "features":           FEATURES,
    "target":             TARGET,
    "best_k":             best_k,
    "station_sequences":  STATION_SEQUENCES,
    "avg_trip_stations":  best_k,
    "test_mae":           round(mae, 2),
    "test_rmse":          round(rmse, 2),
    "test_mape":          round(mape, 2),
    "test_mape_filtered": round(mape_filtrado, 2),
}
meta_path = os.path.join(OUTPUT_DIR, "model_meta.pkl")
with open(meta_path, "wb") as f:
    pickle.dump(meta, f)

print("Archivos guardados:")
print(f"  {model_path}")
print(f"  {meta_path}")
print()
print("Para cargar el modelo en otra sesión:")
print("  import pickle")
print("  from xgboost import XGBRegressor")
print("  model = XGBRegressor()")
print("  model.load_model('xgb_subte_load.json')")
print("  meta  = pickle.load(open('model_meta.pkl', 'rb'))")